In [1]:
import sleap
import numpy as np

# This prevents TensorFlow from allocating all the GPU memory, which leads to issues on
# some GPUs/platforms:
sleap.disable_preallocation()

# This would hide GPUs from the TensorFlow altogether:
# sleap.use_cpu_onl
# Print some info:
sleap.versions()
sleap.system_summary()

SLEAP: 1.4.1
TensorFlow: 2.7.0
Numpy: 1.21.5
Python: 3.7.12
OS: Linux-6.8.0-51-generic-x86_64-with-debian-trixie-sid
GPUs: 2/2 available
  Device: /physical_device:GPU:0
         Available: True
       Initialized: False
     Memory growth: True
  Device: /physical_device:GPU:1
         Available: True
       Initialized: False
     Memory growth: True


In [2]:
import os
import numpy as np
import sys
sys.path.append('/home/mingxiao/Desktop/jelly-sleap/sleap')
sys.path.append('/home/mingxiao/jelly-sleap/sleap')
sys.path.append('/home/mingxiao/')

In [3]:
animal_id = 1
original_labels_dir = f'/home/mingxiao/Desktop/jellyfish/label/animal_{animal_id}_labels.v002.slp'
# original_labels_dir = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v5.slp'
original_labels = sleap.load_file(original_labels_dir)

In [4]:
original_labels

Labels(labeled_frames=8089, videos=1, skeletons=1, tracks=0)

In [5]:
new_labels_dir = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v8.slp'
new_labels = sleap.load_file(new_labels_dir)

In [36]:
new_video = new_labels.videos[0]

In [31]:
simple_tracked_labels_path = '/home/mingxiao/Desktop/jellyfish/label/multifish/predictions/simple-tracking/video_1_simple_tracking.slp'
simple_tracked_labels = sleap.load_file(simple_tracked_labels_path)

In [32]:
simple_tracked_labels

Labels(labeled_frames=3239999, videos=1, skeletons=1, tracks=7)

In [8]:
tb_cnt = len(original_labels.skeletons[0].nodes) - 1
print(f'tb_cnt: {tb_cnt}')
new_skeletons = []
for i in range(1):
    new_skeleton = sleap.Skeleton(name=f'TB')
    new_skeleton.add_node(f'tb')
    new_skeletons.append(new_skeleton)
new_skeletons

tb_cnt: 17


[Skeleton(name='TB', description='None', nodes=['tb'], edges=[], symmetries=[])]

In [9]:
new_labels.skeletons = new_skeletons

In [12]:
original_labels.labeled_frames[0]

LabeledFrame(video=MediaVideo('C:/Users/weiss/OneDrive/Desktop/Concat_and_crop/full_video_1.avi'), frame_idx=0, instances=1)

In [30]:
original_labels.labeled_frames[0].instances[0]

Instance(video=Video(filename=/home/mingxiao/Desktop/jellyfish/video/sleap_full_video_1.mp4, shape=(3240000, 170, 174, 1), backend=MediaVideo), frame_idx=0, points=[tb1_node: (114.4, 26.4)], track=None)

In [26]:
original_labels.labeled_frames[0].instances[0].points[0]

Point(x=114.40617897296741, y=26.368739578785853, visible=True, complete=False)

In [17]:
from copy import copy
copied_instance = copy(original_labels.labeled_frames[0].instances[0])
copied_instance

Instance(video=Video(filename=/home/mingxiao/Desktop/jellyfish/video/sleap_full_video_1.mp4, shape=(3240000, 170, 174, 1), backend=MediaVideo), frame_idx=0, points=[tb1_node: (114.4, 26.4)], track=None)

In [17]:
all_tracks = [sleap.instance.Track(name=f'track_{i}', spawned_on=0) for i in range(17)]
all_tracks

[Track(spawned_on=0, name='track_0'),
 Track(spawned_on=0, name='track_1'),
 Track(spawned_on=0, name='track_2'),
 Track(spawned_on=0, name='track_3'),
 Track(spawned_on=0, name='track_4'),
 Track(spawned_on=0, name='track_5'),
 Track(spawned_on=0, name='track_6'),
 Track(spawned_on=0, name='track_7'),
 Track(spawned_on=0, name='track_8'),
 Track(spawned_on=0, name='track_9'),
 Track(spawned_on=0, name='track_10'),
 Track(spawned_on=0, name='track_11'),
 Track(spawned_on=0, name='track_12'),
 Track(spawned_on=0, name='track_13'),
 Track(spawned_on=0, name='track_14'),
 Track(spawned_on=0, name='track_15'),
 Track(spawned_on=0, name='track_16')]

In [21]:
new_labels.tracks = all_tracks

In [26]:
new_labeled_frames = []
labeled_frame_indices = []
for lf in original_labels.labeled_frames:
    # generate new instances 
    # new_instances = [None for _ in range(tb_cnt)]
    new_instances = []
    labeled_inst = None
    for inst in lf.instances: # each frame has at most 2 instances: labeled and predicted
        if isinstance(inst, sleap.Instance) and not isinstance(inst, sleap.PredictedInstance):
            labeled_inst = inst
            break
    if labeled_inst is None:
        continue
    for old_node, old_point in zip(lf.instances[0].nodes, lf.instances[0].points):
        if old_node.name.lower() == 'mouth':
            continue
        tb_idx = int(old_node.name[2:])
        tb_skeleton = new_skeletons[0]
        point_dict = {f'tb': sleap.instance.Point(x=old_point.x, y=old_point.y)}
        tb_instance = sleap.Instance(skeleton=tb_skeleton, points=point_dict, frame=lf, track=all_tracks[tb_idx-1])
        # new_instances[tb_idx - 1] = tb_instance
        new_instances.append(tb_instance)
    # print the indices where new_instances are None
    if None in new_instances:
        print(f'None indices: {np.where(np.array(new_instances) == None)[0]}')
    assert None not in new_instances, f'some instances are None'
    new_lf = sleap.LabeledFrame(video=new_labels.video, frame_idx=lf.frame_idx, instances=new_instances)
    new_labeled_frames.append(new_lf)
    labeled_frame_indices.append(lf.frame_idx)

In [19]:
new_labels.labeled_frames = new_labeled_frames

In [28]:
new_labels

Labels(labeled_frames=1653, videos=1, skeletons=1, tracks=17)

In [24]:
new_suggestions = []
suggestion_indices = []
for suggestion in original_labels.suggestions:
    new_suggestion = sleap.gui.suggestions.SuggestionFrame(video=new_labels.video, frame_idx=suggestion.frame_idx, group=suggestion.group)
    new_suggestions.append(new_suggestion)
    suggestion_indices.append(suggestion.frame_idx)

In [15]:
new_labels.suggestions = new_suggestions

In [23]:
new_labels_dir = '/home/mingxiao/Desktop/jellyfish/label/multifish/multifish_animal_1_v9.slp'
new_labels.save(new_labels_dir)